In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 09 — Temeller

Bu notebook [09_foundations.md](09_foundations.md) markdown'ının çalıştırılabilir sürümüdür. Axiom vs theorem sınıflandırması, efficient vs foundational mode, `d² = 0`'ın generator-level axiom'dan türetilişi.

## Default engine — her şey axiom

In [ ]:
from gradalg.proof.expansion import default_engine

eng = default_engine()
for d in eng.definitions:
    label = 'theorem' if d.is_theorem else 'axiom'
    print(f'{label:<8} | {d.name}')

## `d_squared_mode="theorem"` — yeniden sınıflandır

In [ ]:
eng_th = default_engine(d_squared_mode='theorem')
for d in eng_th.definitions:
    if d.name == 'd² = 0':
        print('is_theorem    :', d.is_theorem)
        print('has builder?  :', d.theorem_proof_builder() is not None)

## Efficient vs foundational — aynı adım, farklı sub-proof

Efficient mode child taşımaz; foundational mode theorem-sınıfı bir kural tetiklendiğinde generator-level axiom'a atıf iliştirir.

In [ ]:
from gradalg.calculus.invariant_d import default_d
from gradalg.core.registry import PropertyRegistry
from gradalg.core.expr import Symbol, Integer
from gradalg.core.properties import Graded
from gradalg.proof.verifier import prove_equivalence

reg = PropertyRegistry()
omega = Symbol('ω')
reg.declare(omega, Graded(degree=2))
expr = default_d(default_d(omega))

eng_eff = default_engine(registry=reg, mode='efficient',
                         d_squared_mode='theorem')
eng_fnd = default_engine(registry=reg, mode='foundational',
                         d_squared_mode='theorem')

eff = prove_equivalence(expr, Integer(0), registry=reg, engine=eng_eff)
fnd = prove_equivalence(expr, Integer(0), registry=reg, engine=eng_fnd)

print('efficient steps:')
for s in eff.steps:
    print(f'  {s.rule:<15} children={len(s.children)}')
print('foundational steps:')
for s in fnd.steps:
    print(f'  {s.rule:<15} children={len(s.children)}')
    for c in s.children:
        print(f'     ↳ {c.rule}')

## Sub-proof ne söylüyor?

Foundational sub-proof'un tek girdisi `d(df) = 0` — bu paketin primitive kabul ettiği jenerik aksiyom. `d² = 0` tüm form derecelerinde ondan extend ediyor.

In [ ]:
child = fnd.steps[0].children[0]
print('child rule         :', child.rule)
print('child justification:\n  ', child.justification)

## Üç provenance katmanı

Property (sembol), Definition (expansion), Theorem — üçü de `axiom`/`theorem` ayrımı taşır. `Theorem.from_axioms` tek citation olarak makale ispatına iner.

In [ ]:
from gradalg.library import theorem_book

for name in ('poisson_jacobi', 'courant_jacobi_twist'):
    thm = theorem_book.get(name)
    print(name, '->', thm.from_axioms)

## Tutorial serisinin sonu

Dokuz bölüm tamamlandı. Paket artık kendi bracket'iniz, kendi teoreminiz, kendi aksiyom setinizle genişletilebilir bir araç olarak kullanıma hazır.